## Monte Carlo First Visit and Every Visit in GridWorld

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from tqdm.notebook import trange
import matplotlib.colors as mcolors

In [ ]:
class GridworldEnv:
    def __init__(self, grid_size, holes=[]):
        self.grid_size = grid_size
        self.states = [(i, j) for i in range(grid_size) for j in range(grid_size)]
        self.terminal_states = [(0, 0), (grid_size-1, grid_size-1)]
        self.actions = ['up', 'down', 'left', 'right']
        self.holes = holes
        self.rewards = {s: -1 for s in self.states}
        self.rewards[(0, 0)] = 0
        self.rewards[(grid_size-1, grid_size-1)] = 1
        self.gamma = 0.9
        for hole in holes:
            self.rewards[hole] = -np.inf  # Holes have a very negative reward

    def is_terminal(self, state):
        return state in self.terminal_states

    def is_hole(self, state):
        return state in self.holes

    def get_next_state(self, state, action):
        if self.is_terminal(state) or self.is_hole(state):
            return state
        i, j = state
        if action == 'up':
            next_state = (max(i-1, 0), j)
        elif action == 'down':
            next_state = (min(i+1, self.grid_size-1), j)
        elif action == 'left':
            next_state = (i, max(j-1, 0))
        elif action == 'right':
            next_state = (i, min(j+1, self.grid_size-1))
        if self.is_hole(next_state):
            return state  # Stay in the same state if the next state is a hole
        return next_state

    def get_tprob_and_rewards(self, state, action):
        next_state = self.get_next_state(state, action)
        return [(1.0, next_state, self.rewards[next_state])]

    def display_grid(self, filename='gridworld_states.png'):
        fig, ax = plt.subplots()
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                state = (i, j)
                if state in self.terminal_states:
                    color = 'gray'
                elif state in self.holes:
                    color = 'black'
                else:
                    color = 'white'
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, edgecolor='black', facecolor=color))
                ax.text(j + 0.5, i + 0.5, f'({i},{j})', ha='center', va='center')
        ax.set_xlim(0, self.grid_size)
        ax.set_ylim(0, self.grid_size)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.title("Gridworld States")
        plt.gca().invert_yaxis()
        plt.savefig(filename, dpi=300)
        plt.show()

In [ ]:
# Initialize the environment
holes = [(1, 1), (2, 2)]
env = GridworldEnv(grid_size=6, holes=holes)

In [ ]:
env.display_grid()

### Monte Carlo Policy Evaluation

In [ ]:
def random_policy(state):
    return random.choice(['up', 'down', 'left', 'right'])

def generate_episode(env, policy, max_count = 100):
    episode = []
    state = random.choice(env.states)
    while env.is_terminal(state):
        state = random.choice(env.states)
    count = 0
    while not env.is_terminal(state):
        if count < max_count:
            action = policy(state)
            next_state = env.get_next_state(state, action)
            reward = env.rewards[next_state]
            episode.append((state, action, reward))
            state = next_state
            count += 1
        else:
            break
    return episode

def first_visit_mc(env, policy, episodes):
    V = defaultdict(float)
    returns = defaultdict(list)
    for _ in trange(episodes):
        episode = generate_episode(env, policy)
        visited_states = set()
        G = 0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = env.gamma * G + reward
            if state not in visited_states:
                returns[state].append(G)
                V[state] = np.mean(returns[state])
                visited_states.add(state)
    return V

def every_visit_mc(env, policy, episodes):
    V = defaultdict(float)
    returns = defaultdict(list)
    for _ in trange(episodes):
        episode = generate_episode(env, policy)
        G = 0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = env.gamma * G + reward
            returns[state].append(G)
            V[state] = np.mean(returns[state])
    return V

def off_policy_mc(env, target_policy, behavior_policy, episodes):
    V = defaultdict(float)
    C = defaultdict(float)
    for _ in trange(episodes):
        episode = generate_episode(env, behavior_policy)
        G = 0
        W = 1.0
        for t in reversed(range(len(episode))):
            state, action, reward = episode[t]
            G = env.gamma * G + reward
            C[state] += W
            V[state] += (W / C[state]) * (G - V[state])
            if action != target_policy(state):
                break
            W /= 0.25  # Assuming behavior policy is random with probability 0.25
    return V

In [ ]:
# First-visit Monte Carlo
V_first_visit = first_visit_mc(env, random_policy, 10000)

In [ ]:
# Every-visit Monte Carlo
V_every_visit = every_visit_mc(env, random_policy, 10000)

In [ ]:
def plot_value_function(value_function, grid_size, title, filename='', cmap='viridis'):
    values = np.zeros((grid_size, grid_size))
    for (i, j), value in value_function.items():
        values[i, j] = value
    fig, ax = plt.subplots()
    cax = ax.matshow(values, cmap=cmap)
    text_color = 'white'
    if cmap == 'Greys':
        text_color = '0.5'
    for (i, j), value in value_function.items():
        ax.text(j, i, f"{value:.2f}", va='center', ha='center', color=text_color)
    fig.colorbar(cax)
    plt.title(title)
    if filename != '':
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_value_function(V_first_visit, env.grid_size, 'First Visit MC', 'gridworld_ov.png')

In [ ]:
plot_value_function(V_first_visit, env.grid_size, 'First Visit MC', 'gridworld_ov_bw.png', 'Greys')

In [ ]:
plot_value_function(V_every_visit, env.grid_size, 'Every Visit MC', 'gridworld_ev.png')

In [ ]:
plot_value_function(V_every_visit, env.grid_size, 'Every Visit MC', 'gridworld_ev_bw.png', 'Greys')

In [ ]:
# Off-policy Monte Carlo with importance sampling
def target_policy(state):
    return 'right'  # Example deterministic policy

V_off_policy = off_policy_mc(env, target_policy, random_policy, 10000)

In [ ]:
plot_value_function(V_off_policy, env.grid_size, 'Off-policy MC', 'gridworld_op.png')

In [ ]:
plot_value_function(V_off_policy, env.grid_size, 'Off-policy MC', 'gridworld_op_bw.png', 'Greys')